<a href="https://colab.research.google.com/github/FatimaZulfiqarAli-123/Cross_Dialect-Robustnes/blob/main/NLP_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate pandas scikit-learn seaborn matplotlib scipy torch

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy import stats
import torch

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nDialect distribution:")
print(df["dialect"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
models = {
    "AraBERT": "aubmindlab/bert-base-arabertv02",
    "MARBERT": "UBC-NLP/MARBERT",
    "CAMeL": "CAMeL-Lab/bert-base-arabic-camelbert-mix"
}

In [ ]:
device = 0 if torch.cuda.is_available() else -1

pipes = {
    name: pipeline(
        "sentiment-analysis",
        model=model_name,
        device=device
    )
    for name, model_name in models.items()
}

In [ ]:
def predict(pipe, text):
    out = pipe(str(text))[0]

    label = out["label"].upper()
    score = out["score"]

    # safer mapping
    if "POSITIVE" in label or "LABEL_1" in label:
        return 1, score
    else:
        return 0, score

In [ ]:
results = []

for model_name, pipe in pipes.items():
    print(f"Evaluating model: {model_name}")

    for dialect in df["dialect"].unique():
        subset = df[df["dialect"] == dialect]

        y_true = []
        y_pred = []

        for _, row in subset.iterrows():
            pred, _ = predict(pipe, row["text"])
            y_true.append(row["label"])
            y_pred.append(pred)

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")

        results.append({
            "model": model_name,
            "dialect": dialect,
            "accuracy": acc,
            "f1": f1,
            "error_rate": 1 - acc
        })

benchmark_df = pd.DataFrame(results)
benchmark_df

In [ ]:
pivot_acc = benchmark_df.pivot(index="dialect", columns="model", values="accuracy")

plt.figure(figsize=(10,6))
sns.heatmap(pivot_acc, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Cross-Dialect Accuracy Comparison")
plt.show()

In [ ]:
pivot_f1 = benchmark_df.pivot(index="dialect", columns="model", values="f1")

plt.figure(figsize=(10,6))
sns.heatmap(pivot_f1, annot=True, cmap="viridis", fmt=".2f")
plt.title("Cross-Dialect F1 Comparison")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.violinplot(data=benchmark_df, x="model", y="accuracy")
plt.title("Model Robustness Across Dialects")
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=benchmark_df, x="dialect", y="accuracy", hue="model", marker="o")
plt.xticks(rotation=25)
plt.title("Dialect Sensitivity Curve")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=benchmark_df, x="model", y="error_rate")
plt.title("Error Rate Distribution Across Models")
plt.show()

In [ ]:
models_list = benchmark_df["model"].unique()

for i in range(len(models_list)):
    for j in range(i+1, len(models_list)):

        a = benchmark_df[benchmark_df["model"] == models_list[i]]["accuracy"]
        b = benchmark_df[benchmark_df["model"] == models_list[j]]["accuracy"]

        # Welch t-test (better than standard t-test)
        t, p = stats.ttest_ind(a, b, equal_var=False)

        print(f"{models_list[i]} vs {models_list[j]} → p-value: {p:.6f}")

In [ ]:
robustness = benchmark_df.groupby("model")["accuracy"].agg(["mean", "std"]).reset_index()

# Robustness Index (higher = more stable)
robustness["Robustness_Index"] = 1 - (robustness["std"] / robustness["mean"])

robustness

In [ ]:
final_table = benchmark_df.groupby("model").agg({
    "accuracy": "mean",
    "f1": "mean",
    "error_rate": "mean"
}).reset_index()

final_table

In [ ]:
final_table.to_csv("model_summary.csv", index=False)

print("All results saved successfully.")

In [ ]:
plt.savefig("heatmap_accuracy.png", dpi=300, bbox_inches='tight')